[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PatrickJReed/soccer-vision/blob/phase-0-1-bakeoff/examples/bakeoff_tz.ipynb)

# Bake-off candidate: AbdelrahmanAtef01/Tactic_Zone

Runs the Tactic_Zone detection + tracking + team-color stack on the canonical
bake-off clip. **Ignores** Tactic_Zone's event-detection layers (pass/shot/tackle)
— those are out of scope for this bake-off, which only scores detection,
tracking, team-ID, and homography.

**Inputs:** `data/bakeoff_clip.mp4`, `data/bakeoff_clip_corners.json`.

**Outputs:** `data/bakeoff_outputs/tz/trajectories.parquet`,
`data/bakeoff_outputs/tz/annotated.mp4`.

Run on Colab Pro with T4 or L4 GPU.

In [ ]:
from google.colab import userdata
GH_PAT = userdata.get('GITHUB_PAT')  # Set in Colab Secrets (🔑 sidebar) once
!pip install -q "ultralytics>=8.2" "supervision>=0.22"
!git clone https://github.com/AbdelrahmanAtef01/Tactic_Zone /content/tz
import sys; sys.path.insert(0, "/content/tz")
!pip install -q "git+https://${GH_PAT}@github.com/PatrickJReed/soccer-vision.git"
from pathlib import Path

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
INPUT_CLIP = Path('/content/drive/MyDrive/soccer-vision/bakeoff_clip.mp4')
OUT = Path("/content/output/tz")
OUT.mkdir(parents=True, exist_ok=True)
assert INPUT_CLIP.exists(), f"Place bakeoff_clip.mp4 at {INPUT_CLIP}"

In [ ]:
# Tactic_Zone uses YOLOv8x with 4 detection classes: player, goalkeeper, referee, ball.
# Import path determined at clone time — inspect /content/tz/ first.
# Likely structure (verify when running):
#   /content/tz/main.py            — entry-point CLI
#   /content/tz/detector/          — YOLOv8 wrapper
#   /content/tz/tracker/           — ByteTrack or similar
#   /content/tz/team_assignment/   — KMeans-on-shirt-color
#
# TODO when running: replace the placeholders below with the actual TZ imports
# after inspecting the clone tree. Goal is to produce per-frame, per-detection
# rows: (frame, track_id, x_px, y_px, bbox_x1..bbox_y2, class, team, conf).

import cv2

cap = cv2.VideoCapture(str(INPUT_CLIP))
fps = cap.get(cv2.CAP_PROP_FPS)
video_frames = []
while True:
    ok, frame = cap.read()
    if not ok:
        break
    video_frames.append(frame)
cap.release()
print(f"Loaded {len(video_frames)} frames at {fps} fps")

# TODO when running: instantiate TZ's detector + tracker (likely YOLO-based)
# and accumulate per-frame, per-detection rows into `records`.
records: list[dict] = []
# Example skeleton:
# detector = TZDetector("yolov8x_tz.pt")
# tracker = TZTracker()
# for frame_idx, frame in enumerate(video_frames):
#     dets = detector.detect(frame)  # returns boxes, class_ids, confs
#     tracks = tracker.update(dets)
#     for box, tid, cls, conf in tracks:
#         x1, y1, x2, y2 = box
#         cx, cy = (x1 + x2) / 2, y2
#         records.append({...all schema fields...})
print(f"Captured {len(records)} detection rows.")


In [ ]:
# Tactic_Zone has its own KMeans-on-shirt-color team classifier.
# Import path: TODO when running — inspect /content/tz/ for "team_assignment" or similar.
# Convert TZ's team IDs (likely 1/2 or A/B) to schema labels: "own"/"opp"/"ref"/"unknown".
# Ball detections get team="unknown".

# TODO when running:
# from tactic_zone.team_assignment import TeamAssigner  # placeholder import
# assigner = TeamAssigner()
# for r in records:
#     if r["class"] == "player":
#         r["team"] = assigner.assign(video_frames[r["frame"]], r["bbox_x1":"bbox_y2"])
#     elif r["class"] == "referee":
#         r["team"] = "ref"
#     # else: keep "unknown"


In [ ]:
import pandas as pd
from soccer_vision.io.schema import validate_trajectories

df = pd.DataFrame(records)
df = df.astype({"frame": "int64", "track_id": "int64"})
validate_trajectories(df)
df.to_parquet(OUT / "trajectories.parquet")
print(f"Wrote {OUT / 'trajectories.parquet'}: {len(df)} rows")


In [ ]:
# Render boxes + IDs onto the clip
import supervision as sv

cap = cv2.VideoCapture(str(INPUT_CLIP))
fps = cap.get(cv2.CAP_PROP_FPS)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
writer = cv2.VideoWriter(str(OUT / "annotated.mp4"),
                         cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

for frame_idx, group in df.groupby("frame"):
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ok, frame = cap.read()
    if not ok:
        break
    # TODO when running: render boxes + labels (color by team).
    writer.write(frame)

writer.release()
cap.release()
print(f"Wrote {OUT / 'annotated.mp4'}")


In [ ]:
# Download to commit at: data/bakeoff_outputs/tz/{trajectories.parquet,annotated.mp4}
from google.colab import files
files.download(str(OUT / "trajectories.parquet"))
files.download(str(OUT / "annotated.mp4"))
